# ITEC631 – AI-Driven Network Intrusion Detection System
## Sprint 2: Baseline ML Models

**Student:** Hayoung
**Unit:** ITEC631 IT Masters Project Part B, Semester 1 2026

Sprint 2 goal is to train and evaluate three baseline classifiers — Random Forest, XGBoost, and Decision Tree — on the cleaned dataset from Sprint 1. I want to see how each model performs, especially on the rare attack classes, and pick the best one to build the two-stage pipeline on in Sprint 3.

## Setup

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score,
                             precision_score, recall_score)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
RANDOM_STATE = 42

print('libraries loaded')

## 1. Load Processed Data

Loading the train and test sets saved at the end of Sprint 1.
These are already cleaned, normalised, and SMOTE-balanced so I can go straight to training.

In [ ]:
ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    # find the processed_data folder saved from Sprint 1
    # it should be in /kaggle/working if Sprint 1 was run in this session
    # or re-upload it as a separate Kaggle dataset if running fresh
    DATA_PATH = '/kaggle/working/processed_data/'
    if not os.path.exists(DATA_PATH):
        # look for it as an attached dataset
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'train_balanced.csv' in files:
                DATA_PATH = root + '/'
                break
else:
    DATA_PATH = 'processed_data/'

print(f'loading data from: {DATA_PATH}')

train_df = pd.read_csv(DATA_PATH + 'train_balanced.csv')
test_df  = pd.read_csv(DATA_PATH + 'test.csv')

print(f'train set : {train_df.shape[0]:,} rows x {train_df.shape[1]} cols')
print(f'test set  : {test_df.shape[0]:,} rows x {test_df.shape[1]} cols')

In [ ]:
# separate features and labels
X_train = train_df.drop(columns=['label_binary'])
y_train = train_df['label_binary']

X_test  = test_df.drop(columns=['label_binary'])
y_test  = test_df['label_binary']

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'\nclass distribution in training set:')
print(y_train.value_counts().rename({0: 'BENIGN', 1: 'ATTACK'}))
print(f'\nclass distribution in test set:')
print(y_test.value_counts().rename({0: 'BENIGN', 1: 'ATTACK'}))

## 2. Train Baseline Models

Training three classifiers so I can compare them. Using default hyperparameters for now — the goal is just to get a baseline before tuning in later sprints.

- **Random Forest** — ensemble of decision trees, generally robust baseline
- **XGBoost** — gradient boosting, usually strong on tabular data
- **Decision Tree** — simple interpretable model, good for comparison with the reference paper

In [ ]:
# Random Forest
print('training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1   # use all CPU cores
)
rf.fit(X_train, y_train)
print('done')

In [ ]:
# XGBoost
print('training XGBoost...')
xgb = XGBClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='logloss',
)
xgb.fit(X_train, y_train)
print('done')

In [ ]:
# Decision Tree
print('training Decision Tree...')
dt = DecisionTreeClassifier(
    max_depth=20,
    random_state=RANDOM_STATE
)
dt.fit(X_train, y_train)
print('done')

## 3. Evaluation

Evaluating each model on the test set. I'm paying close attention to **recall** —
a missed attack is worse than a false alarm in an IDS context, so recall matters more than precision here.
Overall accuracy can be misleading when classes are imbalanced.

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average='weighted')
    prec   = precision_score(y_test, y_pred, average='weighted')
    rec    = recall_score(y_test, y_pred, average='weighted')

    print(f'\n{name}')
    print('-' * 45)
    print(f'  accuracy  : {acc:.4f}')
    print(f'  precision : {prec:.4f}')
    print(f'  recall    : {rec:.4f}')
    print(f'  f1 score  : {f1:.4f}')
    print(f'\n  classification report:')
    print(classification_report(y_test, y_pred,
                                target_names=['BENIGN', 'ATTACK']))
    return {
        'name': name, 'model': model, 'y_pred': y_pred,
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1
    }

results = []
results.append(evaluate_model('Random Forest',  rf,  X_test, y_test))
results.append(evaluate_model('XGBoost',        xgb, X_test, y_test))
results.append(evaluate_model('Decision Tree',  dt,  X_test, y_test))

## 4. Confusion Matrices

Plotting confusion matrices for all three models side by side so I can see where each one makes mistakes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Confusion Matrices — Binary Classification (BENIGN vs ATTACK)',
             fontsize=13, fontweight='bold')

for ax, res in zip(axes, results):
    cm = confusion_matrix(y_test, res['y_pred'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
                xticklabels=['BENIGN', 'ATTACK'],
                yticklabels=['BENIGN', 'ATTACK'],
                cbar=False)
    ax.set_title(res['name'], fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

    # add raw counts as secondary annotation
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.75, f'({cm[i,j]:,})',
                    ha='center', va='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Comparison

Comparing all three models across the key metrics in one chart.

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1']
labels  = [r['name'] for r in results]
x = np.arange(len(metrics))
width = 0.25
colors = ['#3498db', '#e74c3c', '#2ecc71']

fig, ax = plt.subplots(figsize=(11, 5))

for i, (res, color) in enumerate(zip(results, colors)):
    vals = [res[m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=res['name'],
                  color=color, edgecolor='white', alpha=0.9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1 Score'], fontsize=11)
ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Score')
ax.set_title('Baseline Model Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# summary table
summary = pd.DataFrame([
    {
        'Model'    : r['name'],
        'Accuracy' : f"{r['accuracy']:.4f}",
        'Precision': f"{r['precision']:.4f}",
        'Recall'   : f"{r['recall']:.4f}",
        'F1 Score' : f"{r['f1']:.4f}",
    }
    for r in results
])
print(summary.to_string(index=False))

## 6. Feature Importance

Looking at which features the Random Forest found most useful. This will help in Sprint 4 when I add SHAP explainability.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top20 = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(top20.index, top20.values, color='#3498db', edgecolor='white')
ax.set_title('Random Forest — Top 20 Feature Importances', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')

for bar, val in zip(bars, top20.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Select Best Model

Picking the model with the highest weighted F1 score to carry into Sprint 3 for the two-stage pipeline.
F1 is a better metric than accuracy here because it balances precision and recall across both classes.

In [ ]:
best = max(results, key=lambda r: r['f1'])
print(f'best model by weighted F1: {best["name"]}')
print(f'  f1 score : {best["f1"]:.4f}')
print(f'  accuracy : {best["accuracy"]:.4f}')
print(f'  recall   : {best["recall"]:.4f}')
print(f'\nthis model will be used as Stage 1 classifier in Sprint 3')

## 8. Save Models

Saving all three trained models so I can load them in Sprint 3 without retraining.

In [ ]:
import pickle

os.makedirs('models', exist_ok=True)

model_map = {
    'random_forest'  : rf,
    'xgboost'        : xgb,
    'decision_tree'  : dt,
}

for name, model in model_map.items():
    path = f'models/{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f'saved {path}')

# save best model name for Sprint 3
with open('models/best_model.json', 'w') as f:
    json.dump({'best_model': best['name'], 'f1': best['f1']}, f, indent=2)
print('saved models/best_model.json')

## Sprint 2 Summary

In [ ]:
print('Sprint 2 — Baseline Models complete')
print('='*55)
for r in results:
    print(f"{r['name']:<18} accuracy={r['accuracy']:.4f}  f1={r['f1']:.4f}")
print()
print(f'best model  : {best["name"]}')
print(f'saved to    : models/')
print()
print('next: Sprint 3 — build two-stage pipeline using best model')
print('      Stage 1: BENIGN vs ATTACK')
print('      Stage 2: multi-class attack category classification')

---
## What's Next → Sprint 3: Two-Stage Detection Pipeline

With the best baseline model identified, Sprint 3 will:
1. Build **Stage 1** — binary classifier (BENIGN vs ATTACK) using the best model from Sprint 2
2. Build **Stage 2** — multi-class classifier trained only on attack traffic (DoS/DDoS, PortScan, BruteForce, WebAttack, Botnet, Infiltration)
3. Chain them into a pipeline: traffic goes through Stage 1 first, then flagged attacks go to Stage 2
4. Evaluate the full two-stage system end-to-end